In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


Por qué RobustScaler y no otro scaler? Hay varias opciones para escalar:
StandardScaler — usa la media y desviación típica. Se ve muy afectado por valores extremos (outliers).
MinMaxScaler — escala entre 0 y 1. También sensible a outliers.
RobustScaler — usa la mediana y el rango intercuartílico. Es resistente a outliers.

En nuestro dataset Amount tiene valores extremos (hasta 25.691€). RobustScaler es la mejor opción aquí.

In [2]:
df = pd.read_csv("../data/creditcard.csv")

# Eliminamos la columna Hour que creamos en el EDA
if "Hour" in df.columns:
    df = df.drop(columns=["Hour"])

print(f"Dataset cargado: {df.shape[0]:,} filas, {df.shape[1]} columnas")

Dataset cargado: 284,807 filas, 31 columnas


In [3]:
# Primero separamos X e y antes de tocar nada más
X = df.drop(columns=["Class"])
y = df["Class"]

print(f"X (variables de entrada): {X.shape}")
print(f"y (variable objetivo): {y.shape}")

X (variables de entrada): (284807, 30)
y (variable objetivo): (284807,)


In [4]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]:,} filas")
print(f"Test: {X_test.shape[0]:,} filas")
print(f"\nFraudes en train: {y_train.sum()}")
print(f"Fraudes en test: {y_test.sum()}")

Train: 227,845 filas
Test: 56,962 filas

Fraudes en train: 394
Fraudes en test: 98


In [5]:
# Escalar Amount y Time 
scaler = RobustScaler()

# fit_transform en train — aprende la escala CON train
X_train[["Amount", "Time"]] = scaler.fit_transform(X_train[["Amount", "Time"]])

# transform en test — aplica lo aprendido, sin aprender nada nuevo del test
X_test[["Amount", "Time"]] = scaler.transform(X_test[["Amount", "Time"]])

print("Escalado completado correctamente")
print(f"\nEstadísticas Amount en train después del escalado:")
print(X_train["Amount"].describe().round(3))

Escalado completado correctamente

Estadísticas Amount en train después del escalado:
count    227845.000
mean          0.921
std           3.490
min          -0.306
25%          -0.228
50%           0.000
75%           0.772
max         357.260
Name: Amount, dtype: float64


In [6]:
# SMOTE 
smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Antes de SMOTE:")
print(f"  Legítimas: {(y_train == 0).sum():,}")
print(f"  Fraudes:   {(y_train == 1).sum():,}")

print(f"\nDespués de SMOTE:")
print(f"  Legítimas: {(y_train_sm == 0).sum():,}")
print(f"  Fraudes:   {(y_train_sm == 1).sum():,}")

Antes de SMOTE:
  Legítimas: 227,451
  Fraudes:   394

Después de SMOTE:
  Legítimas: 227,451
  Fraudes:   227,451


In [ ]:
# Guardar datos procesados.
import os

# Creamos carpeta processed dentro de data si no existe
os.makedirs("../data/processesed", exist_ok=True)

# Guardamos los cuatros conjuntos
X_train_sm.to_csv("../data/processed/X_train.csv", index=False)
y_train_sm.to_csv("../data/processed/y_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Datos guardados correctamente en data/processed/")
print(f"\nX_train: {X_train_sm.shape}")
print(f"y_train: {y_train_sm.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_test:  {y_test.shape}")

Datos guardados correctamente en data/processed/

X_train: (454902, 30)
y_train: (454902,)
X_test:  (56962, 30)
y_test:  (56962,)
